In [ ]:
# ==========================================
# 1. 安装与加载必要的 R 包
# ==========================================
# 如果你还没安装 slingshot，请取消下面这行的注释进行安装：
# if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install("slingshot")

library(slingshot)
library(RColorBrewer)
library(fields)

print("Libraries loaded successfully.")

# ==========================================
# 2. 读取数据
# ==========================================
# 直接读取你准备好的 dygen.csv 文件
data <- read.csv('data/dygen.csv')

# 提取时间点 (第1列) 和 降维坐标 (后面的列)
timepoints <- data$samples
# 提取坐标矩阵 (强制转为 matrix 格式，Slingshot 需要)
coords <- as.matrix(data[, -1]) 

print(paste("Loaded data with", nrow(coords), "cells and", ncol(coords), "dimensions."))

# ==========================================
# 3. 对细胞进行聚类 (Slingshot 的前置要求)
# ==========================================
# Slingshot 需要基于 Cluster 来构建最小生成树 (MST)
# 这里我们使用标准的 K-Means 进行聚类，设定为 5 个 cluster (你可以根据 Dyngen 的实际分支情况调整 4~6)
set.seed(42)
cl <- kmeans(coords, centers = 5)$cluster

# 自动寻找起始 Cluster：包含最多 Time = 0 细胞的那个 Cluster
table_time0 <- table(cl[timepoints == 0])
start_cluster <- names(table_time0)[which.max(table_time0)]
print(paste("Automatically identified starting cluster:", start_cluster))

# ==========================================
# 4. 运行 Slingshot 核心算法
# ==========================================
# 直接在降维坐标上运行，指定聚类标签和起始点
sds <- slingshot(coords, clusterLabels = cl, start.clus = start_cluster)

# ==========================================
# 5. 可视化并保存为高清 PNG
# ==========================================
# 记得加载 fields 包，否则最后的 image.plot 会报错
library(fields)

# ---------------------------------------------------------
# 🌟 第一步：打开 PNG 保存设备
# ---------------------------------------------------------
# 设置宽 14 英寸，高 6 英寸，分辨率 300 DPI
png("slingshot_dyngen.png", width = 14, height = 6, units = "in", res = 300)

# ---------------------------------------------------------
# 🌟 第二步：执行画图代码
# ---------------------------------------------------------
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))

# --- 左图：按真实物理时间着色 ---
time_colors <- brewer.pal(max(timepoints) + 1, "Set1")
cell_colors_time <- time_colors[timepoints + 1]

plot(coords[, 1:2], col = cell_colors_time, pch = 16, cex = 0.6,
     main = "Dyngen Cells by True Physical Time", 
     xlab = "Dim 1", ylab = "Dim 2")
legend("topright", legend = paste("Time", 0:max(timepoints)), 
       col = time_colors, pch = 16, bty = "n", cex=0.8)

# --- 右图：Slingshot 拟合的静态主曲线 (双分支完整版) ---

# 1. 提取完整的拟时间矩阵
pseudo_time_matrix <- slingPseudotime(sds)

# 2. 将所有分支的拟时间合并
pseudo_time_combined <- rowMeans(pseudo_time_matrix, na.rm = TRUE)

# 3. 找到有拟时间的有效细胞
valid_cells <- !is.na(pseudo_time_combined)

# 4. 准备颜色映射
pseudo_colors <- colorRampPalette(brewer.pal(11, 'Spectral')[-6])(100) 
cell_colors_pseudo <- rep("lightgray", nrow(coords))

# 5. 将有效细胞的值切分为 100 个区间并映射颜色
color_indices <- as.numeric(cut(pseudo_time_combined[valid_cells], breaks = 100))
cell_colors_pseudo[valid_cells] <- pseudo_colors[color_indices]

# 6. 画散点图
plot(coords[, 1:2], col = cell_colors_pseudo, pch = 16, cex = 0.6,
     main = "Static Trajectory by Slingshot", 
     xlab = "Dim 1", ylab = "Dim 2")

# 7. 叠加 Slingshot 的主曲线骨架
lines(SlingshotDataSet(sds), lwd = 3, col = 'black')

# 添加 Colorbar
image.plot(zlim = c(0, 1), 
           col = pseudo_colors, 
           legend.only = TRUE, 
           smallplot = c(0.85, 0.88, 0.4, 0.8), 
           axis.args = list(at=c(0, 1), labels=c("Low", "High")), 
           legend.args = list(text="Pseudotime", side=4, line=2))

# ---------------------------------------------------------
# 🌟 第三步：关闭设备，将图片写入磁盘
# ---------------------------------------------------------
dev.off()

Warning message:
"package 'slingshot' was built under R version 4.4.1"
Loading required package: princurve

Warning message:
"package 'princurve' was built under R version 4.4.3"
Loading required package: TrajectoryUtils

Warning message:
"package 'TrajectoryUtils' was built under R version 4.4.1"
Loading required package: SingleCellExperiment

Warning message:
"package 'SingleCellExperiment' was built under R version 4.4.1"
Loading required package: SummarizedExperiment

Warning message:
"package 'SummarizedExperiment' was built under R version 4.4.1"
Loading required package: MatrixGenerics

Warning message:
"package 'MatrixGenerics' was built under R version 4.4.2"
Loading required package: matrixStats

Warning message:
"package 'matrixStats' was built under R version 4.4.3"

Attaching package: 'MatrixGenerics'


The following objects are masked from 'package:matrixStats':

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCump

[1] "Libraries loaded successfully."
[1] "Loaded data with 728 cells and 5 dimensions."
[1] "Automatically identified starting cluster: 2"


agg_record_1852181441 
                    2

In [ ]:
# ==========================================
# 1. 安装与加载必要的 R 包
# ==========================================
# 如果你还没安装 slingshot，请取消下面这行的注释进行安装：
# if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install("slingshot")

library(slingshot)
library(RColorBrewer)

print("Libraries loaded successfully.")

# ==========================================
# 2. 读取数据
# ==========================================
# 直接读取你准备好的 dygen.csv 文件
data <- read.csv('data/simulation_gene_data.csv')

# 提取时间点 (第1列) 和 降维坐标 (后面的列)
timepoints <- data$samples
# 提取坐标矩阵 (强制转为 matrix 格式，Slingshot 需要)
coords <- as.matrix(data[, -1]) 

print(paste("Loaded data with", nrow(coords), "cells and", ncol(coords), "dimensions."))

# ==========================================
# 3. 对细胞进行聚类 (Slingshot 的前置要求)
# ==========================================
# Slingshot 需要基于 Cluster 来构建最小生成树 (MST)
# 这里我们使用标准的 K-Means 进行聚类，设定为 5 个 cluster (你可以根据 Dyngen 的实际分支情况调整 4~6)
set.seed(42)
cl <- kmeans(coords, centers = 5)$cluster

# 自动寻找起始 Cluster：包含最多 Time = 0 细胞的那个 Cluster
table_time0 <- table(cl[timepoints == 0])
start_cluster <- names(table_time0)[which.max(table_time0)]
print(paste("Automatically identified starting cluster:", start_cluster))

# ==========================================
# 4. 运行 Slingshot 核心算法
# ==========================================
# 直接在降维坐标上运行，指定聚类标签和起始点
sds <- slingshot(coords, clusterLabels = cl, start.clus = 5)

# ==========================================
# 5. 可视化并保存为高清 PNG
# ==========================================
# 记得加载 fields 包，否则最后的 image.plot 会报错
library(fields)

# ---------------------------------------------------------
# 🌟 第一步：打开 PNG 保存设备
# ---------------------------------------------------------
# 设置宽 14 英寸，高 6 英寸，分辨率 300 DPI
png("slingshot_simulation.png", width = 14, height = 6, units = "in", res = 300)

# ---------------------------------------------------------
# 🌟 第二步：执行画图代码
# ---------------------------------------------------------
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))

# --- 左图：按真实物理时间着色 ---
time_colors <- brewer.pal(max(timepoints) + 1, "Set1")
cell_colors_time <- time_colors[timepoints + 1]

plot(coords[, 1:2], col = cell_colors_time, pch = 16, cex = 0.6,
     main = "Simulation Cells by True Physical Time", 
     xlab = "Dim 1", ylab = "Dim 2")
legend("topright", legend = paste("Time", 0:max(timepoints)), 
       col = time_colors, pch = 16, bty = "n", cex=0.8)

# --- 右图：Slingshot 拟合的静态主曲线 (双分支完整版) ---

# 1. 提取完整的拟时间矩阵
pseudo_time_matrix <- slingPseudotime(sds)

# 2. 将所有分支的拟时间合并
pseudo_time_combined <- rowMeans(pseudo_time_matrix, na.rm = TRUE)

# 3. 找到有拟时间的有效细胞
valid_cells <- !is.na(pseudo_time_combined)

# 4. 准备颜色映射
pseudo_colors <- colorRampPalette(brewer.pal(11, 'Spectral')[-6])(100) 
cell_colors_pseudo <- rep("lightgray", nrow(coords))

# 5. 将有效细胞的值切分为 100 个区间并映射颜色
color_indices <- as.numeric(cut(pseudo_time_combined[valid_cells], breaks = 100))
cell_colors_pseudo[valid_cells] <- pseudo_colors[color_indices]

# 6. 画散点图
plot(coords[, 1:2], col = cell_colors_pseudo, pch = 16, cex = 0.6,
     main = "Static Trajectory by Slingshot", 
     xlab = "Dim 1", ylab = "Dim 2")

# 7. 叠加 Slingshot 的主曲线骨架
lines(SlingshotDataSet(sds), lwd = 3, col = 'black')

# 添加 Colorbar
image.plot(zlim = c(0, 1), 
           col = pseudo_colors, 
           legend.only = TRUE, 
           smallplot = c(0.85, 0.88, 0.4, 0.8), 
           axis.args = list(at=c(0, 1), labels=c("Low", "High")), 
           legend.args = list(text="Pseudotime", side=4, line=2))

# ---------------------------------------------------------
# 🌟 第三步：关闭设备，将图片写入磁盘
# ---------------------------------------------------------
dev.off()

[1] "Libraries loaded successfully."


[1] "Loaded data with 3031 cells and 2 dimensions."
[1] "Automatically identified starting cluster: 3"


agg_record_674280576 
                   2

In [ ]:
# ==========================================
# 1. 安装与加载必要的 R 包
# ==========================================
# 如果你还没安装 slingshot，请取消下面这行的注释进行安装：
# if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install("slingshot")

library(slingshot)
library(RColorBrewer)

print("Libraries loaded successfully.")

# ==========================================
# 2. 读取数据
# ==========================================
# 直接读取你准备好的 dygen.csv 文件
data <- read.csv('data/simulation_gene_data.csv')

# 提取时间点 (第1列) 和 降维坐标 (后面的列)
timepoints <- data$samples
# 提取坐标矩阵 (强制转为 matrix 格式，Slingshot 需要)
coords <- as.matrix(data[, -1]) 

print(paste("Loaded data with", nrow(coords), "cells and", ncol(coords), "dimensions."))

# ==========================================
# 3. 对细胞进行聚类 (Slingshot 的前置要求)
# ==========================================
# Slingshot 需要基于 Cluster 来构建最小生成树 (MST)
# 这里我们使用标准的 K-Means 进行聚类，设定为 5 个 cluster (你可以根据 Dyngen 的实际分支情况调整 4~6)
set.seed(42)
cl <- kmeans(coords, centers = 5)$cluster

# 自动寻找起始 Cluster：包含最多 Time = 0 细胞的那个 Cluster
table_time0 <- table(cl[timepoints == 0])
start_cluster <- names(table_time0)[which.max(table_time0)]
print(paste("Automatically identified starting cluster:", start_cluster))

# ==========================================
# 4. 运行 Slingshot 核心算法
# ==========================================
# 直接在降维坐标上运行，指定聚类标签和起始点
sds <- slingshot(coords, clusterLabels = cl, start.clus = start_cluster)

# ==========================================
# 5. 可视化并保存为高清 PNG
# ==========================================
# 记得加载 fields 包，否则最后的 image.plot 会报错
library(fields)

# ---------------------------------------------------------
# 🌟 第一步：打开 PNG 保存设备
# ---------------------------------------------------------
# 设置宽 14 英寸，高 6 英寸，分辨率 300 DPI
png("slingshot_simulation2.png", width = 14, height = 6, units = "in", res = 300)

# ---------------------------------------------------------
# 🌟 第二步：执行画图代码
# ---------------------------------------------------------
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))

# --- 左图：按真实物理时间着色 ---
time_colors <- brewer.pal(max(timepoints) + 1, "Set1")
cell_colors_time <- time_colors[timepoints + 1]

plot(coords[, 1:2], col = cell_colors_time, pch = 16, cex = 0.6,
     main = "Simulation Cells by True Physical Time", 
     xlab = "Dim 1", ylab = "Dim 2")
legend("topright", legend = paste("Time", 0:max(timepoints)), 
       col = time_colors, pch = 16, bty = "n", cex=0.8)

# --- 右图：Slingshot 拟合的静态主曲线 (双分支完整版) ---

# 1. 提取完整的拟时间矩阵
pseudo_time_matrix <- slingPseudotime(sds)

# 2. 将所有分支的拟时间合并
pseudo_time_combined <- rowMeans(pseudo_time_matrix, na.rm = TRUE)

# 3. 找到有拟时间的有效细胞
valid_cells <- !is.na(pseudo_time_combined)

# 4. 准备颜色映射
pseudo_colors <- colorRampPalette(brewer.pal(11, 'Spectral')[-6])(100) 
cell_colors_pseudo <- rep("lightgray", nrow(coords))

# 5. 将有效细胞的值切分为 100 个区间并映射颜色
color_indices <- as.numeric(cut(pseudo_time_combined[valid_cells], breaks = 100))
cell_colors_pseudo[valid_cells] <- pseudo_colors[color_indices]

# 6. 画散点图
plot(coords[, 1:2], col = cell_colors_pseudo, pch = 16, cex = 0.6,
     main = "Static Trajectory by Slingshot", 
     xlab = "Dim 1", ylab = "Dim 2")

# 7. 叠加 Slingshot 的主曲线骨架
lines(SlingshotDataSet(sds), lwd = 3, col = 'black')

# 添加 Colorbar
image.plot(zlim = c(0, 1), 
           col = pseudo_colors, 
           legend.only = TRUE, 
           smallplot = c(0.85, 0.88, 0.4, 0.8), 
           axis.args = list(at=c(0, 1), labels=c("Low", "High")), 
           legend.args = list(text="Pseudotime", side=4, line=2))

# ---------------------------------------------------------
# 🌟 第三步：关闭设备，将图片写入磁盘
# ---------------------------------------------------------
dev.off()

[1] "Libraries loaded successfully."
[1] "Loaded data with 3031 cells and 2 dimensions."
[1] "Automatically identified starting cluster: 3"


agg_record_674280576 
                   2